# Tydzień 8: Wykrywanie anomalii (*Anomaly Detection*) 🕵️‍♀️

| | |
|---|---|
| **Rozdział książki** | 5. *Anomaly Detection* |
| **Czas zajęć** | 90 minut |
| **Dane** | `wholesale` – roczne wydatki 440 klientów hurtowni w Portugalii (wbudowane w PyCaret) |
| **Nowe funkcje PyCaret** | moduł `pycaret.anomaly`: `create_model('lof')`, `assign_model()`, `plot_model(..., 'umap')` |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. rozumieć, czym są **obserwacje odstające** (*outliers*) i do czego służy **wykrywanie anomalii**,
2. odróżniać anomalie **jednowymiarowe** (*univariate*) od **wielowymiarowych** (*multivariate*),
3. znać intuicję stojącą za modelem **Local Outlier Factor (LOF)**,
4. umieć wykryć anomalie w PyCaret, oddzielić je od „normalnych” danych i sprawdzić efekt.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–15 min | 📚 Teoria: anomalie, LOF, UMAP |
| 15–65 min | 👣 Krok po kroku |
| 65–85 min | 🧑‍💻 Ćwiczenie: drugi detektor – Isolation Forest – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Co to jest anomalia? 🧥🏖️
**Anomalia** (obserwacja odstająca, *outlier*) to obserwacja, która **wyraźnie różni się od większości** danych –
jak ktoś w zimowym płaszczu na plaży w lipcu.

Gdzie się to przydaje?
- 💳 **banki** – wykrywanie podejrzanych transakcji (np. skradziona karta),
- 🏭 **fabryki** – wykrywanie awarii maszyn na podstawie odczytów czujników,
- 🌐 **sieci komputerowe** – wykrywanie włamań,
- 🧹 **czyszczenie danych** – usuwanie błędów pomiarowych przed trenowaniem innych modeli.

To zwykle **uczenie nienadzorowane**: nie mamy etykiet „to jest oszustwo”, więc model sam szuka tego, co nietypowe.

## Anomalie jedno- i wielowymiarowe 📏⚖️
- **Jednowymiarowa** (*univariate*): nietypowa wartość **jednej** cechy, np. wzrost 2,30 m.
- **Wielowymiarowa** (*multivariate*): każda cecha osobno wygląda normalnie, ale ich **połączenie** jest dziwne.
  Wzrost 1,95 m – OK. Waga 45 kg – OK (np. u nastolatka). Ale **1,95 m i 45 kg u tej samej osoby**? To już anomalia!

Dziś szukamy anomalii **wielowymiarowych** – patrzymy na wszystkie cechy klienta naraz.

## Local Outlier Factor (LOF) – „czy mieszkasz na odludziu?” 🏘️
LOF porównuje **gęstość okolicy** punktu z gęstością okolic jego sąsiadów:
- jeśli punkt mieszka w „zatłoczonym centrum”, tak jak jego sąsiedzi → LOF ≈ **1** → punkt normalny (*inlier*),
- jeśli punkt mieszka **samotnie na odludziu**, a jego najbliżsi sąsiedzi – w zatłoczonym centrum → LOF **wyraźnie > 1** → anomalia.

$$\mathrm{LOF}_k(A) = \frac{\text{średnia gęstość okolic sąsiadów } A}{\text{gęstość okolicy } A}$$

Parametr **`fraction`** to **szacowany odsetek anomalii** w danych (domyślnie 5%). Model oznaczy jako anomalie
5% punktów z najwyższym wynikiem.

## UMAP – „mapa metra” 🚇
Tak jak PCA, **UMAP** zamienia wiele wymiarów na 2, żeby dało się je narysować. Działa jak **mapa metra**: nie zachowuje
dokładnych odległości, ale dobrze pokazuje, **kto jest czyim sąsiadem**. Dzięki temu skupiska i „samotniki” są dobrze widoczne.

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Biblioteki do tabel i wykresów
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# get_data – dane; moduł WYKRYWANIA ANOMALII PyCaret
from pycaret.datasets import get_data
from pycaret.anomaly import *

### Krok 2: Wczytujemy dane klientów hurtowni 🛒
Każdy wiersz to **jeden klient** hurtowni (np. restauracja albo sklep). Kolumny:

| kolumna | znaczenie |
|---|---|
| `Channel` | kanał: `Horeca` (hotele, restauracje, kawiarnie) albo `Retail` (sklepy detaliczne) |
| `Region` | region: `Lisbon` (Lizbona), `Oporto` (Porto), `Other` (inne) |
| `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, `Delicassen` | **roczne wydatki** na: świeże produkty, nabiał, artykuły spożywcze, mrożonki, detergenty i papier, delikatesy |

In [ ]:
# Listy kolumn liczbowych i kategorycznych
numeric = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
categorical = ['Channel', 'Region']

In [ ]:
# Wczytujemy dane (verbose=False – bez automatycznego podglądu)
data = get_data('wholesale', verbose=False)
data.head()

🔎 Kolumny `Channel` i `Region` są zakodowane liczbami (1, 2, 3). Dla czytelności wykresów zamieńmy je z powrotem na nazwy.

In [ ]:
# Słownik: kolumna -> {stara wartość: nowa wartość}
replace_dict = {'Channel': {1: 'Horeca', 2: 'Retail'},
                'Region': {1: 'Lisbon', 2: 'Oporto', 3: 'Other'}}

In [ ]:
# replace() podmienia wartości według słownika
data = data.replace(replace_dict)
data.head()

### Krok 3: EDA – szukamy „podejrzanych” wartości 🔍

In [ ]:
# Histogramy wydatków w każdej kategorii produktów
data[numeric].hist(bins=30, figsize=(12, 10), grid=False)
plt.show()

🔎 Wszystkie rozkłady są **mocno prawostronnie skośne**: większość klientów wydaje mało, ale nieliczni – ogromnie dużo.
Ci nieliczni to kandydaci na anomalie.

In [ ]:
# Wydatki na artykuły spożywcze z podziałem na kanał sprzedaży
sns.histplot(data=data, x='Grocery', hue='Channel', multiple='stack')
plt.show()

In [ ]:
# Wykresy pudełkowe wszystkich wydatków (orient='h' – poziomo); xlim ogranicza oś do 60 000
sns.boxplot(data=data[numeric], orient='h')
plt.xlim(0, 60000)
plt.show()

🔎 Kropki na prawo od „wąsów” to **jednowymiarowe** wartości odstające – są w każdej kolumnie!
Ale uwaga: klient z dużymi wydatkami na *jeden* typ produktów nie musi być anomalią. Nas interesuje nietypowa **kombinacja** wydatków.

In [ ]:
# Liczba klientów w każdej kombinacji kanał × region; unstack() zamienia wynik w tabelkę
group = data.groupby(['Channel', 'Region']).size().unstack()
group

In [ ]:
# Ta sama tabelka jako wykres słupkowy
group.plot(kind='bar', figsize=(8, 5))
plt.show()

In [ ]:
# Korelacje wydatków (tylko kolumny liczbowe)
sns.heatmap(data[numeric].corr().round(2), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

🔎 Silna korelacja `Grocery` – `Detergents_Paper` (0,92): kto kupuje dużo artykułów spożywczych, ten kupuje też dużo detergentów (typowe dla sklepów).

In [ ]:
# Wykresy rozrzutu dla 4 kategorii wydatków, kolor = kanał sprzedaży
cols = ['Milk', 'Grocery', 'Frozen', 'Detergents_Paper']
sns.pairplot(data, vars=cols, hue='Channel', plot_kws={'s': 15})
plt.show()

### Krok 4: Inicjalizacja środowiska – `setup()`
Tak jak w grupowaniu, **nie podajemy zmiennej celu** – nie wiemy z góry, kto jest anomalią.

In [ ]:
# PyCaret sam rozpozna kolumny kategoryczne (Channel, Region) i zakoduje je metodą one-hot
anomaly = setup(data, session_id=8477)

### Krok 5: Model Local Outlier Factor 🏘️

In [ ]:
# 'lof' = Local Outlier Factor; fraction=0.05 – zakładamy, że ok. 5% klientów to anomalie
model = create_model('lof', fraction=0.05)

In [ ]:
# assign_model dodaje dwie kolumny: Anomaly (1 = anomalia, 0 = normalny) i Anomaly_Score (wynik LOF)
data_assigned = assign_model(model)
data_assigned.head(10)

In [ ]:
# Ilu klientów uznano za anomalie (1), a ilu za normalnych (0)?
data_assigned['Anomaly'].value_counts()

In [ ]:
# query() wybiera wiersze spełniające warunek – osobno anomalie i osobno klienci normalni
data_outliers = data_assigned.query('Anomaly == 1')
data_inliers = data_assigned.query('Anomaly == 0')

In [ ]:
# Podejrzani klienci – zwróć uwagę na ogromne wydatki w niektórych kategoriach
data_outliers.head(10)

In [ ]:
# Średni wynik LOF w obu grupach: u normalnych ok. 1, u anomalii wyraźnie więcej
data_assigned.groupby('Anomaly')['Anomaly_Score'].mean()

### Krok 6: Ocena modelu – czy dane „uspokoiły się” po usunięciu anomalii? 📉
Nie mamy prawdziwych etykiet, więc ocenimy model pośrednio: jeśli usunęliśmy prawdziwe anomalie, rozkłady powinny stać się
**mniej skośne**.

In [ ]:
# Skośność wydatków w CAŁYM zbiorze
data[numeric].skew().round(2)

In [ ]:
# Skośność wydatków tylko u klientów NORMALNYCH (bez anomalii)
data_inliers[numeric].skew().round(2)

In [ ]:
# Średnia skośność przed i po usunięciu anomalii
print("Średnia skośność – wszyscy klienci:", round(data[numeric].skew().mean(), 2))
print("Średnia skośność – bez anomalii:   ", round(data_inliers[numeric].skew().mean(), 2))

🔎 Skośność **wyraźnie spadła** – usunięcie zaledwie 5% klientów sprawiło, że dane są dużo bardziej „typowe”.
To znak, że model znalazł naprawdę nietypowe obserwacje.

### Krok 7: Wizualizacja anomalii 🎨

In [ ]:
# Wykresy rozrzutu z kolorem według etykiety Anomaly; diag_kind='hist' – histogramy na przekątnej
sns.pairplot(data_assigned, vars=cols, hue='Anomaly', diag_kind='hist', plot_kws={'s': 15})
plt.show()

🔎 Większość punktów **daleko od „chmury”** została oznaczona jako anomalie (kolor dla `Anomaly = 1`).

In [ ]:
# Wykres UMAP: wszystkie cechy "spłaszczone" do 2 wymiarów; kolor = anomalia (interaktywny – najedź myszką na punkt)
plot_model(model, plot='umap')

### Krok 8: Zapis modelu 💾

In [ ]:
# Zapisujemy detektor anomalii (średnik ukrywa długi wydruk)
import os
os.makedirs('../modele', exist_ok=True)
save_model(model, '../modele/anomaly_detection_model');

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Drugi detektor – Isolation Forest** 🌲

**Isolation Forest** (las izolacji) działa jak gra w „20 pytań”: losowo dzieli dane („czy wydatki na mleko > 5000?”).
Nietypowego klienta da się **odizolować** od reszty bardzo małą liczbą pytań – i to właśnie zdradza anomalię.

- **a)** Zbuduj model `'iforest'` z parametrem `fraction=0.05` i przypisz etykiety (`assign_model`) do zmiennej `data_iforest`.
- **b)** Ilu klientów Isolation Forest uznał za anomalie?
- **c)** Ilu klientów uznały za anomalie **oba** modele (LOF i Isolation Forest)?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `create_model('iforest', fraction=0.05)`, a potem `assign_model(...)`.
- b) `data_iforest['Anomaly'].value_counts()`.
- c) Numery wierszy anomalii: `data_iforest.query('Anomaly == 1').index`. Wspólne elementy dwóch zbiorów
  znajdziesz operacją `set(a) & set(b)` (część wspólna), a liczbę elementów – funkcją `len()`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Ile anomalii zakładamy? – parametr `fraction`
Nie wiemy, ilu nietypowych klientów naprawdę jest w danych – 5% to tylko założenie.

- **a)** Zbuduj model LOF z `fraction=0.10` i przypisz etykiety (`data_lof10`). Ilu klientów uznano teraz za anomalie?
- **b)** Policz średnią skośność wydatków klientów **normalnych** i porównaj z wynikiem dla 5% (ok. 2,16).
- **c)** Czy wszystkie anomalie z modelu 5% są też anomaliami w modelu 10%? Dlaczego?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Kopiuj kod z kroków 5–6, zmieniając tylko `fraction`.
- Zbiór numerów wierszy: `set(tabela.query('Anomaly == 1').index)`.
- Czy zbiór A zawiera się w B? W Pythonie: `A <= B` (wynik `True`/`False`).

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Portret nietypowego klienta
Wróć do modelu 5% (tabela `data_assigned`) i przygotuj krótki „raport” dla działu sprzedaży.

- **a)** W którym **kanale** (`Channel`) i **regionie** (`Region`) odsetek anomalii jest najwyższy?
- **b)** Porównaj **średnie wydatki** anomalii i klientów normalnych w każdej kategorii. **Ile razy** więcej wydają anomalie?
- **c)** Pokaż 5 klientów z **najwyższym** wynikiem `Anomaly_Score`. Co jest w nich nietypowego?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Odsetek anomalii w grupie = średnia z kolumny 0/1: `data_assigned.groupby('Channel')['Anomaly'].mean()`.
- Średnie wydatki: `data_assigned.groupby('Anomaly')[numeric].mean()` – wiersz `1` to anomalie, wiersz `0` – normalni.
  Stosunek: `srednie.loc[1] / srednie.loc[0]`.
- 5 największych: `.nlargest(5, 'Anomaly_Score')`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **Anomalia** to obserwacja wyraźnie różna od większości; szukamy jej zwykle bez etykiet (uczenie nienadzorowane).
- Anomalia **wielowymiarowa** to nietypowe **połączenie** wartości, a nie tylko jedna duża liczba.
- **LOF** porównuje gęstość okolicy punktu z gęstością okolic sąsiadów (LOF ≈ 1 – normalny, LOF ≫ 1 – anomalia).
- Parametr `fraction` = zakładany odsetek anomalii; `assign_model()` dodaje kolumny `Anomaly` i `Anomaly_Score`.
- Po usunięciu anomalii dane stają się mniej skośne – to pośredni sposób oceny modelu.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| wykrywanie anomalii | anomaly detection | szukanie nietypowych obserwacji |
| obserwacja odstająca | outlier | punkt wyraźnie różny od reszty |
| obserwacja typowa | inlier | „normalny” punkt |
| anomalia jednowymiarowa | univariate outlier | nietypowa wartość jednej cechy |
| anomalia wielowymiarowa | multivariate outlier | nietypowe połączenie wielu cech |
| lokalny współczynnik odstawania | Local Outlier Factor (LOF) | porównanie gęstości okolic punktów |
| gęstość | density | jak „tłoczno” jest wokół punktu |
| odsetek anomalii | fraction / contamination | zakładany procent anomalii w danych |
| las izolacji | Isolation Forest | izolowanie punktów losowymi podziałami |
| UMAP | Uniform Manifold Approximation and Projection | redukcja wymiarów zachowująca sąsiedztwo |

➡️ **Za tydzień:** **przetwarzanie języka naturalnego (NLP)** – chmury słów i wykrywanie tematów w artykułach BBC 📰.